<a href="https://colab.research.google.com/github/Kalana-Lakshan/ML-Self-Learning/blob/main/ML_practice_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [103]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor

In [104]:
train_data = pd.read_csv('/content/drive/MyDrive/Lab 2 Regression/train.csv')
test_data = pd.read_csv('/content/drive/MyDrive/Lab 2 Regression/test.csv')
sample_sub_data = pd.read_csv('/content/drive/MyDrive/Lab 2 Regression/sample_submission.csv')

In [105]:
train_data.head()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,composition_label_0,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,...,time_signature_0,duration_ms_1,harmonic_scale_0,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0,target
0,76339,0.482850,1.169231,80.018,0.0201,Country Stuff (feat. Jake Owen),1.0,0.789,154586.0,NaN,...,4.0,161853.0,7.0,4.0,NaN,0.607,7.0,0.7250,0.000000,74
1,80006,0.267862,1.321321,147.966,0.3340,Solitude,6.0,0.715,46874.0,15.0,...,4.0,155619.0,1.0,4.0,0.843,0.783,4.0,NaN,0.043200,2
2,83501,0.242606,1.285319,142.980,0.1110,BDFFRNT (Saved from Conformity),4.0,NaN,264665.0,7.0,...,4.0,209378.0,6.0,4.0,NaN,0.211,10.0,0.6020,0.000000,35
3,81530,0.426400,1.279435,123.063,0.1960,Headlights (feat. Ilsey),5.0,0.685,209208.0,5.0,...,4.0,219043.0,11.0,4.0,0.702,0.369,NaN,0.8200,0.000335,70
4,60534,0.000000,0.974906,132.722,0.0811,Afraid,6.0,0.856,215346.0,5.0,...,4.0,258893.0,1.0,0.0,0.000,0.631,1.0,0.0221,0.000000,78


In [106]:
test_data.head()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,composition_label_0,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,...,emotional_resonance_2,time_signature_0,duration_ms_1,harmonic_scale_0,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0
0,25174,0.600480,1.543590,124.008,0.0729,Dr.Q,1.0,0.763,23032.0,4.0,...,0.834,4.0,253987.0,4.0,4.0,0.604,0.2050,0.0,0.720,0.165000
1,38453,NaN,0.722420,129.942,0.0105,Start A Party,11.0,0.801,215466.0,20.0,...,0.216,4.0,267626.0,5.0,4.0,0.881,0.2610,1.0,0.496,0.000000
2,29013,0.461916,0.757962,83.000,0.2700,Sombras - Live,2.0,0.561,252261.0,44.0,...,0.546,4.0,226626.0,0.0,4.0,0.555,0.0555,0.0,0.846,0.002760
3,57463,0.144236,0.923977,183.991,0.1210,Tennis Court,2.0,NaN,198907.0,23.0,...,0.337,4.0,234286.0,0.0,4.0,0.674,0.4040,7.0,0.428,0.000194
4,51264,0.629832,1.473795,201.277,0.0610,La Cumbia Del Lazo,10.0,0.716,158720.0,NaN,...,0.966,4.0,188520.0,8.0,4.0,0.511,0.9620,10.0,0.652,0.000115


In [107]:
train_data.isnull().sum().sum()/(train_data.shape[0]*train_data.shape[1])*100

np.float64(5.485740196106664)

In [108]:
train_data.isnull().sum()

,0
id,0
emotional_charge_2,2442
groove_efficiency_1,180
beat_frequency_1,386
organic_texture_2,383
...,...
emotional_resonance_0,1546
harmonic_scale_2,4467
intensity_index_2,693
instrumental_density_0,709


In [109]:
train_data['publication_timestamp'] = pd.to_datetime(train_data['publication_timestamp'])
test_data['publication_timestamp'] = pd.to_datetime(test_data['publication_timestamp'])

train_data['year'] = train_data['publication_timestamp'].dt.year
train_data['month'] = train_data['publication_timestamp'].dt.month
train_data['day'] = train_data['publication_timestamp'].dt.day

test_data['year'] = test_data['publication_timestamp'].dt.year
test_data['month'] = test_data['publication_timestamp'].dt.month
test_data['day'] = test_data['publication_timestamp'].dt.day

train_data.drop('publication_timestamp',axis = 1,inplace = True)
test_data.drop('publication_timestamp',axis = 1,inplace = True)


In [110]:
#identifying numerical columns and categorical columns seperately
num_cols = train_data.select_dtypes(include = np.number).columns.drop("target")
cat_cols = train_data.select_dtypes(include = 'object').columns

In [111]:
len(num_cols)

55

In [112]:
len(cat_cols)

8

In [113]:
cat_cols

Index(['composition_label_0', 'composition_label_1', 'weekday_of_release',
       'season_of_release', 'lunar_phase', 'creator_collective',
       'composition_label_2', 'track_identifier'],
      dtype='object')

In [114]:
#missing values handling
train_data[num_cols] = train_data[num_cols].fillna(train_data[num_cols].median())
train_data[cat_cols] = train_data[cat_cols].fillna(train_data[num_cols].mode()).astype(str)

In [115]:
test_data[num_cols] = test_data[num_cols].fillna(train_data[num_cols].median())
test_data[cat_cols] = test_data[cat_cols].fillna(train_data[num_cols].mode()).astype(str)

In [116]:
train_data.isnull().sum().sum()

np.int64(0)

In [117]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61609 entries, 0 to 61608
Data columns (total 64 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          61609 non-null  int64  
 1   emotional_charge_2          61609 non-null  float64
 2   groove_efficiency_1         61609 non-null  float64
 3   beat_frequency_1            61609 non-null  float64
 4   organic_texture_2           61609 non-null  float64
 5   composition_label_0         61609 non-null  object 
 6   harmonic_scale_1            61609 non-null  float64
 7   intensity_index_0           61609 non-null  float64
 8   duration_ms_0               61609 non-null  float64
 9   album_name_length           61609 non-null  float64
 10  beat_frequency_0            61609 non-null  float64
 11  beat_frequency_2            61609 non-null  float64
 12  artist_count                61609 non-null  float64
 13  composition_label_1         616

In [118]:
#Dividing into nominal and ordinal categories
nominal_cat_cols = ['composition_label_0', 'composition_label_1','weekday_of_release', 'season_of_release', 'lunar_phase','creator_collective', 'composition_label_2', 'track_identifier']



In [119]:
for col in nominal_cat_cols:
  print(train_data[col].nunique())

21710
22931
8
5
5
15140
22170
22992


In [120]:
#Feature Selection
cols_to_drop = ['composition_label_0', 'composition_label_1','creator_collective', 'composition_label_2', 'track_identifier']

train_data.drop(cols_to_drop,axis = 1,inplace=True)
test_data.drop(cols_to_drop,axis = 1,inplace=True)

In [121]:
train_data.head()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,beat_frequency_0,...,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0,target,year,month,day
0,76339,0.482850,1.169231,80.018,0.0201,1.0,0.789,154586.0,14.0,95.992,...,4.0,0.630,0.607,7.0,0.7250,0.000000,74,2021.0,6.0,4.0
1,80006,0.267862,1.321321,147.966,0.3340,6.0,0.715,46874.0,15.0,148.076,...,4.0,0.843,0.783,4.0,0.6460,0.043200,2,2019.0,7.0,1.0
2,83501,0.242606,1.285319,142.980,0.1110,4.0,0.633,264665.0,7.0,124.738,...,4.0,0.630,0.211,10.0,0.6020,0.000000,35,2014.0,11.0,18.0
3,81530,0.426400,1.279435,123.063,0.1960,5.0,0.685,209208.0,5.0,119.893,...,4.0,0.702,0.369,5.0,0.8200,0.000335,70,2015.0,9.0,25.0
4,60534,0.000000,0.974906,132.722,0.0811,6.0,0.856,215346.0,5.0,118.006,...,0.0,0.000,0.631,1.0,0.0221,0.000000,78,2006.0,1.0,1.0


In [122]:
test_data.head()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,beat_frequency_0,...,harmonic_scale_0,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0,year,month,day
0,25174,0.600480,1.543590,124.008,0.0729,1.0,0.763,23032.0,4.0,124.262,...,4.0,4.0,0.604,0.2050,0.0,0.720,0.165000,2011.0,10.0,21.0
1,38453,0.291060,0.722420,129.942,0.0105,11.0,0.801,215466.0,20.0,132.070,...,5.0,4.0,0.881,0.2610,1.0,0.496,0.000000,2016.0,8.0,12.0
2,29013,0.461916,0.757962,83.000,0.2700,2.0,0.561,252261.0,44.0,119.893,...,0.0,4.0,0.555,0.0555,0.0,0.846,0.002760,2011.0,1.0,1.0
3,57463,0.144236,0.923977,183.991,0.1210,2.0,0.633,198907.0,23.0,90.019,...,0.0,4.0,0.674,0.4040,7.0,0.428,0.000194,2017.0,6.0,14.0
4,51264,0.629832,1.473795,201.277,0.0610,10.0,0.716,158720.0,14.0,197.715,...,8.0,4.0,0.511,0.9620,10.0,0.652,0.000115,1988.0,1.0,1.0


In [123]:
final_nominal_cols = ['weekday_of_release', 'season_of_release', 'lunar_phase']

In [124]:
train_data.isnull().sum().sum()

np.int64(0)

In [125]:
#encoding
onehotencoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_train_data = onehotencoder.fit_transform(train_data[final_nominal_cols])
encoded_test_data = onehotencoder.transform(test_data[final_nominal_cols])

encoded_cols = onehotencoder.get_feature_names_out(final_nominal_cols)
encoded_train_df = pd.DataFrame(encoded_train_data, columns=encoded_cols, index = train_data.index)
encoded_test_df = pd.DataFrame(encoded_test_data, columns=encoded_cols, index = test_data.index)

# Drop original categorical columns and concatenate with encoded dataframes
train_data = pd.concat([train_data.drop(final_nominal_cols, axis=1), encoded_train_df], axis=1)
test_data = pd.concat([test_data.drop(final_nominal_cols, axis=1), encoded_test_df], axis=1)

In [126]:
train_data.isnull().sum().sum()

np.int64(0)

In [127]:
#scaling
scaler = StandardScaler()
train_data[num_cols] = scaler.fit_transform(train_data[num_cols])
test_data[num_cols] = scaler.transform(test_data[num_cols])

In [128]:
train_data.isnull().sum().sum()

np.int64(0)

In [129]:
X = train_data.drop(train_data.columns[-1],axis = 1)
y = train_data[train_data.columns[-1]]

In [130]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [131]:
model = RandomForestRegressor(
    n_estimators = 10,
    max_depth = 5,
    random_state = 42,
    n_jobs = -1
)
model.fit(X_train,y_train)

RandomForestRegressor(max_depth=5, n_estimators=10, n_jobs=-1, random_state=42)

In [132]:
y_predict = model.predict(X_test)

In [134]:
print("MSE ",mean_squared_error(y_test,y_predict))
print("MAE ",mean_absolute_error(y_test,y_predict))
print("R2 ",r2_score(y_test,y_predict))

MSE  0.0
MAE  0.0
R2  1.0


In [ ]:
output = pd.DataFrame({"id":test_data["id"],"target":model.predict(test_data)})
output.to_csv("submission1.csv")